# Build Project in Colab + Smoke Test

This notebook writes the project source files into Google Drive and runs a **small smoke test** before the full multilingual run.

That gives you:
- persistence across Colab sessions
- a reproducible codebase
- a clean way to debug before spending GPU/CPU time on the whole corpus


In [ ]:
%%capture
!pip install -q numpy scipy pandas polars soundfile librosa matplotlib plotly pyyaml tqdm joblib faster-whisper silero-vad onnxruntime-gpu transformers


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/indic_audio_filter"
PROJECT_DIR = f"{DRIVE_DIR}/project_repo"
DATA_DIR = f"{DRIVE_DIR}/data"
OUTPUT_DIR = f"{DRIVE_DIR}/outputs"

import os, pathlib
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Project directory:", PROJECT_DIR)


In [ ]:
from pathlib import Path
full = Path(PROJECT_DIR) / Path(r"requirements-colab.txt")
full.parent.mkdir(parents=True, exist_ok=True)
full.write_text('numpy\nscipy\npandas\npolars\nsoundfile\nlibrosa\nmatplotlib\nplotly\npyyaml\ntqdm\njoblib\nfaster-whisper\nsilero-vad\nonnxruntime-gpu\ntransformers\nhuggingface_hub\n', encoding='utf-8')
print("Wrote", full)


In [ ]:
from pathlib import Path
full = Path(PROJECT_DIR) / Path(r"configs/default.yaml")
full.parent.mkdir(parents=True, exist_ok=True)
full.write_text('audio:\n  target_sr: 16000\n  mono: true\n  max_duration_sec: 30.0\n\nmodules:\n  enable_vad: true\n  enable_quality_proxy: true\n  enable_asr_bonus: true\n  enable_langid_bonus: true\n\nthresholds:\n  reject:\n    min_duration_sec: 0.50\n    max_duration_sec: 30.0\n    min_rms_db: -45.0\n    max_clipping_ratio: 0.020\n    max_silence_ratio: 0.85\n    min_speech_ratio: 0.15\n    min_quality_proxy: 0.25\n  review:\n    max_clipping_ratio: 0.005\n    max_silence_ratio: 0.60\n    min_speech_ratio: 0.30\n    min_quality_proxy: 0.45\n    min_asr_confidence: -1.25\n\nscoring:\n  weights:\n    signal: 0.30\n    vad: 0.25\n    quality: 0.25\n    asr: 0.15\n    language: 0.05\n  keep_min: 70.0\n  review_min: 45.0\n', encoding='utf-8')
print("Wrote", full)


In [ ]:
from pathlib import Path
full = Path(PROJECT_DIR) / Path(r"src/__init__.py")
full.parent.mkdir(parents=True, exist_ok=True)
full.write_text('', encoding='utf-8')
print("Wrote", full)


In [ ]:
from pathlib import Path
full = Path(PROJECT_DIR) / Path(r"src/config.py")
full.parent.mkdir(parents=True, exist_ok=True)
full.write_text("import yaml\ndef load_config(path):\n    with open(path, 'r', encoding='utf-8') as f:\n        return yaml.safe_load(f)\n", encoding='utf-8')
print("Wrote", full)


In [ ]:
from pathlib import Path
full = Path(PROJECT_DIR) / Path(r"src/io_utils.py")
full.parent.mkdir(parents=True, exist_ok=True)
full.write_text("import json\nfrom pathlib import Path\nimport polars as pl\n\ndef ensure_dir(path):\n    p = Path(path); p.mkdir(parents=True, exist_ok=True); return p\n\ndef read_manifest(path):\n    path = Path(path)\n    if path.is_dir():\n        rows = []\n        for file in sorted(path.rglob('*.jsonl')):\n            rows.extend(pl.read_ndjson(file).to_dicts())\n        return rows\n    return pl.read_ndjson(path).to_dicts()\n\ndef write_jsonl(rows, path):\n    path = Path(path); ensure_dir(path.parent)\n    with open(path, 'w', encoding='utf-8') as f:\n        for row in rows:\n            f.write(json.dumps(row, ensure_ascii=False) + '\\n')\n\ndef write_csv(rows, path):\n    path = Path(path); ensure_dir(path.parent)\n    pl.DataFrame(rows).write_csv(path)\n\ndef write_parquet(rows, path):\n    path = Path(path); ensure_dir(path.parent)\n    pl.DataFrame(rows).write_parquet(path)\n", encoding='utf-8')
print("Wrote", full)


In [ ]:
from pathlib import Path
full = Path(PROJECT_DIR) / Path(r"src/audio_utils.py")
full.parent.mkdir(parents=True, exist_ok=True)
full.write_text("import numpy as np, librosa, soundfile as sf\n\ndef load_and_standardize_audio(path, target_sr=16000, mono=True):\n    try:\n        wav, sr = sf.read(path, always_2d=False)\n        wav = wav.astype(np.float32)\n        if wav.ndim > 1 and mono:\n            wav = wav.mean(axis=1)\n        if sr != target_sr:\n            wav = librosa.resample(wav, orig_sr=sr, target_sr=target_sr)\n            sr = target_sr\n        if not np.isfinite(wav).all():\n            wav = np.nan_to_num(wav)\n        return {'ok': True, 'waveform': wav.astype(np.float32), 'sample_rate': int(sr), 'duration_sec': float(len(wav)/sr) if sr > 0 else 0.0}\n    except Exception as e:\n        return {'ok': False, 'error': str(e)}\n", encoding='utf-8')
print("Wrote", full)


In [ ]:
from pathlib import Path
full = Path(PROJECT_DIR) / Path(r"src/metrics/__init__.py")
full.parent.mkdir(parents=True, exist_ok=True)
full.write_text('', encoding='utf-8')
print("Wrote", full)


In [ ]:
from pathlib import Path
full = Path(PROJECT_DIR) / Path(r"src/metrics/basic.py")
full.parent.mkdir(parents=True, exist_ok=True)
full.write_text("import numpy as np, librosa\n\ndef rms_db(wav):\n    eps = 1e-10\n    return float(20*np.log10(np.sqrt(np.mean(np.square(wav))+eps)+eps))\n\ndef clipping_ratio(wav, threshold=0.99):\n    return float(np.mean(np.abs(wav) >= threshold))\n\ndef silence_ratio_amp(wav, frame_length=400, hop_length=160, db_threshold=-40.0):\n    if len(wav) < frame_length:\n        return 1.0 if rms_db(wav) < db_threshold else 0.0\n    frames = librosa.util.frame(wav, frame_length=frame_length, hop_length=hop_length).T\n    frame_rms = np.sqrt(np.mean(np.square(frames), axis=1)+1e-10)\n    frame_db = 20*np.log10(frame_rms+1e-10)\n    return float(np.mean(frame_db < db_threshold))\n\ndef compute_basic_metrics(wav, sr):\n    centroid = librosa.feature.spectral_centroid(y=wav, sr=sr)\n    flatness = librosa.feature.spectral_flatness(y=wav)\n    zcr = librosa.feature.zero_crossing_rate(y=wav)\n    return {\n        'rms_db': rms_db(wav),\n        'peak_abs': float(np.max(np.abs(wav))) if len(wav) else 0.0,\n        'clipping_ratio': clipping_ratio(wav),\n        'silence_ratio_amp': silence_ratio_amp(wav),\n        'dc_offset': float(np.abs(np.mean(wav))) if len(wav) else 0.0,\n        'zero_crossing_rate': float(np.mean(zcr)),\n        'spectral_centroid_mean': float(np.mean(centroid)),\n        'spectral_flatness_mean': float(np.mean(flatness)),\n    }\n", encoding='utf-8')
print("Wrote", full)


In [ ]:
from pathlib import Path
full = Path(PROJECT_DIR) / Path(r"src/metrics/vad.py")
full.parent.mkdir(parents=True, exist_ok=True)
full.write_text("import numpy as np\ntry:\n    from silero_vad import load_silero_vad, get_speech_timestamps\n    _HAS_SILERO = True\nexcept Exception:\n    _HAS_SILERO = False\n\nclass VADWrapper:\n    def __init__(self, sr=16000):\n        self.sr = sr\n        self.model = load_silero_vad() if _HAS_SILERO else None\n\n    def compute(self, wav):\n        if len(wav) == 0:\n            return {'speech_duration_sec': 0.0, 'speech_ratio': 0.0, 'num_speech_segments': 0, 'max_silence_sec': 0.0, 'mean_speech_segment_sec': 0.0, 'vad_backend': 'none'}\n        if self.model is not None:\n            ts = get_speech_timestamps(wav, self.model, sampling_rate=self.sr)\n            segs = [(d['start']/self.sr, d['end']/self.sr) for d in ts]\n            backend = 'silero'\n        else:\n            frame = int(0.03*self.sr); hop = int(0.01*self.sr)\n            segs = []; backend = 'energy_fallback'; in_speech = False; start = 0.0\n            thr = max(0.015, 0.5 * float(np.sqrt(np.mean(wav**2))+1e-8))\n            for i in range(0, max(1, len(wav)-frame), hop):\n                rms = float(np.sqrt(np.mean(wav[i:i+frame]**2)+1e-10))\n                sp = rms > thr; t = i/self.sr\n                if sp and not in_speech:\n                    start = t; in_speech = True\n                elif not sp and in_speech:\n                    segs.append((start, t)); in_speech = False\n            if in_speech:\n                segs.append((start, len(wav)/self.sr))\n        total = len(wav)/self.sr\n        speech_dur = sum(max(0.0, e-s) for s,e in segs)\n        silences=[]; prev=0.0\n        for s,e in segs:\n            silences.append(max(0.0, s-prev)); prev=e\n        silences.append(max(0.0, total-prev))\n        seg_lens=[max(0.0,e-s) for s,e in segs]\n        return {\n            'speech_duration_sec': float(speech_dur),\n            'speech_ratio': float(speech_dur/total) if total>0 else 0.0,\n            'num_speech_segments': int(len(segs)),\n            'max_silence_sec': float(max(silences) if silences else 0.0),\n            'mean_speech_segment_sec': float(np.mean(seg_lens) if seg_lens else 0.0),\n            'vad_backend': backend\n        }\n", encoding='utf-8')
print("Wrote", full)


In [ ]:
from pathlib import Path
full = Path(PROJECT_DIR) / Path(r"src/metrics/quality_proxy.py")
full.parent.mkdir(parents=True, exist_ok=True)
full.write_text("import numpy as np\n\ndef compute_quality_proxy(basic, vad):\n    rms_norm = np.clip((basic['rms_db'] + 50.0) / 35.0, 0.0, 1.0)\n    clip_penalty = np.clip(1.0 - (basic['clipping_ratio'] / 0.02), 0.0, 1.0)\n    silence_penalty = np.clip(1.0 - basic['silence_ratio_amp'], 0.0, 1.0)\n    speech_score = np.clip(vad['speech_ratio'] / 0.6, 0.0, 1.0)\n    flatness_penalty = np.clip(1.0 - abs(basic['spectral_flatness_mean'] - 0.2) / 0.4, 0.0, 1.0)\n    overall = float(np.mean([rms_norm, clip_penalty, silence_penalty, speech_score, flatness_penalty]))\n    return {'quality_proxy': overall, 'quality_proxy_mos_like': float(1.0 + 4.0*overall)}\n", encoding='utf-8')
print("Wrote", full)


In [ ]:
from pathlib import Path
full = Path(PROJECT_DIR) / Path(r"src/metrics/asr_bonus.py")
full.parent.mkdir(parents=True, exist_ok=True)
full.write_text("try:\n    from faster_whisper import WhisperModel\n    _HAS_FW = True\nexcept Exception:\n    _HAS_FW = False\n\nclass ASRBonus:\n    def __init__(self, model_size='tiny', device='auto', compute_type='float16'):\n        self.ok = False\n        self.model = None\n        if _HAS_FW:\n            try:\n                self.model = WhisperModel(model_size, device=device, compute_type=compute_type)\n                self.ok = True\n            except Exception:\n                self.ok = False\n\n    def transcribe(self, audio_path):\n        if not self.ok:\n            return {'asr_text':'', 'asr_avg_logprob':None, 'asr_detected_language':None, 'asr_confidence_proxy':None, 'asr_backend':'unavailable'}\n        segments, info = self.model.transcribe(audio_path, beam_size=1, vad_filter=True)\n        segs = list(segments)\n        text = ' '.join((s.text or '').strip() for s in segs).strip()\n        vals = [s.avg_logprob for s in segs if getattr(s, 'avg_logprob', None) is not None]\n        avg = float(sum(vals)/len(vals)) if vals else None\n        conf = None if avg is None else max(0.0, min(1.0, (avg + 2.0)/2.0))\n        return {'asr_text': text, 'asr_avg_logprob': avg, 'asr_detected_language': getattr(info,'language',None), 'asr_confidence_proxy': conf, 'asr_backend':'faster_whisper'}\n", encoding='utf-8')
print("Wrote", full)


In [ ]:
from pathlib import Path
full = Path(PROJECT_DIR) / Path(r"src/metrics/langid_bonus.py")
full.parent.mkdir(parents=True, exist_ok=True)
full.write_text("def compute_language_match(expected_language, predicted_language):\n    if not expected_language or not predicted_language:\n        return {'language_match': None, 'language_score': None}\n    match = expected_language.lower()[:2] == predicted_language.lower()[:2]\n    return {'language_match': bool(match), 'language_score': 1.0 if match else 0.0}\n", encoding='utf-8')
print("Wrote", full)


In [ ]:
from pathlib import Path
full = Path(PROJECT_DIR) / Path(r"src/scoring.py")
full.parent.mkdir(parents=True, exist_ok=True)
full.write_text("def _normalize(value, lo, hi):\n    if value is None:\n        return None\n    if hi == lo:\n        return 0.0\n    return max(0.0, min(1.0, (value - lo) / (hi - lo)))\n\ndef apply_hard_rules(record, cfg):\n    t = cfg['thresholds']['reject']; reasons = []\n    if record.get('io_error'): reasons.append('io_error')\n    if record.get('duration_sec',0) < t['min_duration_sec']: reasons.append('too_short')\n    if record.get('duration_sec',0) > t['max_duration_sec']: reasons.append('too_long')\n    if record.get('rms_db',0) < t['min_rms_db']: reasons.append('low_energy')\n    if record.get('clipping_ratio',0) > t['max_clipping_ratio']: reasons.append('high_clipping')\n    if record.get('silence_ratio_amp',0) > t['max_silence_ratio']: reasons.append('high_silence_ratio')\n    if record.get('speech_ratio',0) < t['min_speech_ratio']: reasons.append('low_speech_ratio')\n    if record.get('quality_proxy',1.0) < t['min_quality_proxy']: reasons.append('low_quality_proxy')\n    return reasons\n\ndef compute_scores(record, cfg):\n    signal_parts = [_normalize(record.get('rms_db'), -45, -10), 1.0 - _normalize(record.get('clipping_ratio'), 0.0, 0.02), 1.0 - _normalize(record.get('silence_ratio_amp'), 0.0, 0.85)]\n    signal_score = sum(x for x in signal_parts if x is not None) / max(1, len([x for x in signal_parts if x is not None]))\n    vad_parts = [_normalize(record.get('speech_ratio'), 0.15, 0.75), 1.0 - _normalize(record.get('max_silence_sec'), 0.5, 12.0)]\n    vad_score = sum(x for x in vad_parts if x is not None) / max(1, len([x for x in vad_parts if x is not None]))\n    quality_score = record.get('quality_proxy', 0.0)\n    asr_score = record.get('asr_confidence_proxy') if record.get('asr_confidence_proxy') is not None else 0.5\n    language_score = record.get('language_score') if record.get('language_score') is not None else 0.5\n    w = cfg['scoring']['weights']\n    final = 100.0 * (w['signal']*signal_score + w['vad']*vad_score + w['quality']*quality_score + w['asr']*asr_score + w['language']*language_score)\n    return {'signal_score': round(signal_score*100,2), 'vad_score': round(vad_score*100,2), 'quality_score': round(quality_score*100,2), 'asr_score': round(asr_score*100,2), 'language_score_pct': round(language_score*100,2), 'final_score': round(final,2)}\n\ndef final_decision(record, hard_reasons, cfg):\n    reasons = list(hard_reasons)\n    if hard_reasons:\n        return 'reject', reasons\n    rt = cfg['thresholds']['review']\n    if record.get('clipping_ratio',0) > rt['max_clipping_ratio']: reasons.append('review_clipping')\n    if record.get('silence_ratio_amp',0) > rt['max_silence_ratio']: reasons.append('review_silence')\n    if record.get('speech_ratio',1) < rt['min_speech_ratio']: reasons.append('review_low_speech')\n    if record.get('quality_proxy',1) < rt['min_quality_proxy']: reasons.append('review_low_quality')\n    if record.get('asr_avg_logprob') is not None and record['asr_avg_logprob'] < rt['min_asr_confidence']: reasons.append('review_low_asr_confidence')\n    if record.get('language_match') is False: reasons.append('review_language_mismatch')\n    score = record.get('final_score',0.0)\n    if score >= cfg['scoring']['keep_min'] and not reasons: return 'keep', reasons\n    if score < cfg['scoring']['review_min']:\n        reasons.append('low_final_score')\n        return 'reject', reasons\n    return 'review', reasons\n", encoding='utf-8')
print("Wrote", full)


In [ ]:
from pathlib import Path
full = Path(PROJECT_DIR) / Path(r"src/pipeline.py")
full.parent.mkdir(parents=True, exist_ok=True)
full.write_text("from pathlib import Path\nfrom tqdm import tqdm\nfrom .audio_utils import load_and_standardize_audio\nfrom .metrics.basic import compute_basic_metrics\nfrom .metrics.vad import VADWrapper\nfrom .metrics.quality_proxy import compute_quality_proxy\nfrom .metrics.asr_bonus import ASRBonus\nfrom .metrics.langid_bonus import compute_language_match\nfrom .scoring import apply_hard_rules, compute_scores, final_decision\n\nclass FilteringPipeline:\n    def __init__(self, cfg, enable_asr=True):\n        self.cfg = cfg\n        self.vad = VADWrapper(sr=cfg['audio']['target_sr'])\n        self.asr = ASRBonus(model_size='tiny', device='auto', compute_type='float16') if enable_asr and cfg['modules'].get('enable_asr_bonus', True) else None\n\n    def process_one(self, row):\n        sample_id = row.get('sample_id') or Path(row.get('audio_filepath') or row.get('audio_path')).stem\n        audio_path = row.get('audio_filepath') or row.get('audio_path')\n        language = row.get('language') or row.get('lang')\n        record = {'sample_id': sample_id, 'audio_path': audio_path, 'language': language}\n        loaded = load_and_standardize_audio(audio_path, target_sr=self.cfg['audio']['target_sr'], mono=self.cfg['audio']['mono'])\n        if not loaded['ok']:\n            record.update({'io_error': loaded['error'], 'decision': 'reject', 'reason_codes': ['io_error']})\n            return record\n        wav = loaded['waveform']; sr = loaded['sample_rate']\n        record['duration_sec'] = round(loaded['duration_sec'], 4)\n        record.update(compute_basic_metrics(wav, sr))\n        vad_metrics = self.vad.compute(wav); record.update(vad_metrics)\n        record.update(compute_quality_proxy(record, vad_metrics))\n        hard = apply_hard_rules(record, self.cfg)\n        if self.asr is not None and not hard:\n            record.update(self.asr.transcribe(audio_path))\n            record.update(compute_language_match(language, record.get('asr_detected_language')))\n        else:\n            record.update({'asr_text':'', 'asr_avg_logprob':None, 'asr_detected_language':None, 'asr_confidence_proxy':None, 'asr_backend':'skipped'})\n            record.update(compute_language_match(language, None))\n        record.update(compute_scores(record, self.cfg))\n        decision, reasons = final_decision(record, hard, self.cfg)\n        record['decision'] = decision; record['reason_codes'] = reasons\n        return record\n\n    def run(self, rows):\n        return [self.process_one(row) for row in tqdm(rows, desc='Filtering audio')]\n", encoding='utf-8')
print("Wrote", full)


In [ ]:
from pathlib import Path
full = Path(PROJECT_DIR) / Path(r"src/visualize.py")
full.parent.mkdir(parents=True, exist_ok=True)
full.write_text("from pathlib import Path\nimport polars as pl\nimport matplotlib.pyplot as plt\n\ndef create_summary_plots(metrics_path, output_dir):\n    out = Path(output_dir); out.mkdir(parents=True, exist_ok=True)\n    pdf = pl.read_parquet(metrics_path).to_pandas()\n    plt.figure(figsize=(8,4)); pdf['final_score'].hist(bins=40); plt.title('Final quality score distribution'); plt.xlabel('Final score'); plt.ylabel('Count'); plt.tight_layout(); plt.savefig(out/'score_distribution.png', dpi=150); plt.close()\n    plt.figure(figsize=(8,4)); pdf['decision'].value_counts().plot(kind='bar'); plt.title('Decision breakdown'); plt.xlabel('Decision'); plt.ylabel('Count'); plt.tight_layout(); plt.savefig(out/'decision_breakdown.png', dpi=150); plt.close()\n    if 'language' in pdf.columns and pdf['language'].notna().any():\n        lang_keep = pdf.groupby('language')['decision'].apply(lambda s: (s == 'keep').mean()*100).sort_values(ascending=False)\n        plt.figure(figsize=(10,4)); lang_keep.plot(kind='bar'); plt.title('Keep rate by language'); plt.ylabel('Keep rate (%)'); plt.tight_layout(); plt.savefig(out/'keep_rate_by_language.png', dpi=150); plt.close()\n", encoding='utf-8')
print("Wrote", full)


In [ ]:
from pathlib import Path
full = Path(PROJECT_DIR) / Path(r"src/main.py")
full.parent.mkdir(parents=True, exist_ok=True)
full.write_text("import argparse, json\nfrom pathlib import Path\nfrom .config import load_config\nfrom .io_utils import read_manifest, write_jsonl, write_csv, write_parquet, ensure_dir\nfrom .pipeline import FilteringPipeline\nfrom .visualize import create_summary_plots\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument('--manifest', required=True)\n    ap.add_argument('--config', default='configs/default.yaml')\n    ap.add_argument('--output_dir', required=True)\n    ap.add_argument('--disable_asr', action='store_true')\n    ap.add_argument('--limit', type=int, default=None)\n    args = ap.parse_args()\n    cfg = load_config(args.config)\n    out = Path(args.output_dir); ensure_dir(out)\n    rows = read_manifest(args.manifest)\n    if args.limit: rows = rows[:args.limit]\n    pipeline = FilteringPipeline(cfg, enable_asr=not args.disable_asr)\n    results = pipeline.run(rows)\n    write_parquet(results, out/'metrics.parquet')\n    write_csv(results, out/'metrics.csv')\n    write_jsonl(results, out/'decisions.jsonl')\n    for label in ['keep','review','reject']:\n        write_jsonl([r for r in results if r.get('decision') == label], out/f'{label}_manifest.jsonl')\n    summary = {'total_samples': len(results), 'keep': sum(r['decision']=='keep' for r in results), 'review': sum(r['decision']=='review' for r in results), 'reject': sum(r['decision']=='reject' for r in results)}\n    summary['keep_rate_pct'] = round(100.0 * summary['keep'] / max(1, summary['total_samples']), 2)\n    with open(out/'summary.json','w',encoding='utf-8') as f: json.dump(summary,f,indent=2)\n    create_summary_plots(str(out/'metrics.parquet'), str(out/'plots'))\n    print(json.dumps(summary, indent=2))\n\nif __name__ == '__main__':\n    main()\n", encoding='utf-8')
print("Wrote", full)


In [ ]:
from pathlib import Path
full = Path(PROJECT_DIR) / Path(r"README.md")
full.parent.mkdir(parents=True, exist_ok=True)
full.write_text('# Indic Audio Filtering Pipeline (Colab-first)\n\nThis repository is a Colab-first solution for the **Audio Filtering Pipeline for Indic Speech** assignment.\n\n## Core features\n- IndicVoices setup via the provided dataset script\n- Basic signal metrics: duration, RMS dB, clipping ratio, silence ratio, DC offset, ZCR, spectral features\n- VAD-based metrics: speech ratio, speech duration, max silence, number of speech segments\n- Quality proxy layer for no-reference filtering\n- Bonus layer: ASR confidence and language-consistency checks\n- Final decisions: `keep`, `review`, `reject`\n- Deliverables: CSV, JSONL, Parquet, plots, summary JSON\n\n## Notebook order\n1. `notebooks/01_dataset_setup.ipynb`\n2. `notebooks/02_build_project_and_smoke_test.ipynb`\n3. `notebooks/03_run_full_pipeline_and_visualize.ipynb`\n4. `notebooks/04_bonus_asr_lid_analysis.ipynb`\n', encoding='utf-8')
print("Wrote", full)


In [ ]:
from pathlib import Path
full = Path(PROJECT_DIR) / Path(r"COLAB_PROJECT_GUIDE.md")
full.parent.mkdir(parents=True, exist_ok=True)
full.write_text('# Colab project guide\n\n## Run order\n1. Dataset setup\n2. Build project and smoke test\n3. Full pipeline run and visualizations\n4. Bonus ASR/LID analysis\n\n## Outputs\n- metrics.csv\n- metrics.parquet\n- decisions.jsonl\n- keep_manifest.jsonl\n- review_manifest.jsonl\n- reject_manifest.jsonl\n- plots/*.png\n- summary.json\n', encoding='utf-8')
print("Wrote", full)


In [ ]:
from pathlib import Path
full = Path(PROJECT_DIR) / Path(r"RESEARCH_NOTES.md")
full.parent.mkdir(parents=True, exist_ok=True)
full.write_text("# Research notes\n\nThis project is designed around the assignment's requirement to filter **raw speech recordings at scale**.\n\nPractical principle:\n- use cheap signal heuristics first\n- estimate speech structure with VAD\n- add a no-reference quality layer\n- optionally use ASR/LID for borderline samples\n", encoding='utf-8')
print("Wrote", full)


In [ ]:
import sys, glob, subprocess
sys.path.insert(0, PROJECT_DIR)
manifests = sorted(glob.glob(f"{DATA_DIR}/manifests/**/*.jsonl", recursive=True))
assert manifests, "No manifests found. Run Notebook 1 first."
test_manifest = manifests[0]
with open(test_manifest, "r", encoding="utf-8") as f:
    lines = []
    for i, line in enumerate(f):
        if i >= 10: break
        lines.append(line)
with open("/content/test_10.jsonl", "w", encoding="utf-8") as f:
    f.writelines(lines)

cmd = [
    "python", "-m", "src.main",
    "--manifest", "/content/test_10.jsonl",
    "--config", f"{PROJECT_DIR}/configs/default.yaml",
    "--output_dir", "/content/smoke_output"
]
res = subprocess.run(cmd, cwd=PROJECT_DIR, capture_output=True, text=True)
print("Return code:", res.returncode)
print(res.stdout[-2000:])
print(res.stderr[-2000:])


In [ ]:
import os
for root, _, files in os.walk("/content/smoke_output"):
    for f in files:
        print(os.path.join(root, f))
